# Apply marine quality control functions on MAROB data

In [1]:
%load_ext autoreload
%autoreload 2

## Import python libraries

In [5]:
import requests

In [6]:
import pandas as pd

In [7]:
from marine_qc import (
    do_position_check, 
    do_date_check, 
    do_time_check, 
    do_missing_value_check,
    do_hard_limit_check, 
    do_sst_freeze_check,
    do_supersaturation_check,
    do_wind_consistency_check,
    do_multiple_individual_check,
)

## Load remote data into a pandas DataFrame

The daily data is stored remotely as json dictionaries. The data will be overwritten in the night with the data from the previous day. 

In [8]:
url = "https://oflks472.dwd.de:3443/api/marob_yesterday"

In [9]:
res =  requests.get(url, verify=False)

C:\Users\llierham\mobaxterm\.venvs\mp_py\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oflks472.dwd.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [10]:
data = res.json()

In [11]:
df = pd.DataFrame(data)
df

,IID,marob_id,kennung,geogr_laenge,geogr_laenge_flag,geogr_breite,geogr_breite_flag,stationshoehe_msl,barometerhoehe_msl,messzeit,...,luftdruck_reduziert,luftdruck_reduziert_flag,wassertemperatur,wassertemperatur_flag,messtiefe,sensorhoehe_was_ff,windrichtung,windrichtung_flag,windgeschwindigkeit,windgeschwindigkeit_flag
0,10384,438974296,2AZY7HU,-128.90,3.0,53.4,3.0,1.1,NaN,2026-03-23T00:00:00,...,1019.0,None,10.6,None,NaN,8.3,110.0,None,9.3,None
1,10384,438982166,2AZY7HU,-128.90,3.0,53.4,3.0,1.1,NaN,2026-03-23T01:00:00,...,1018.3,None,NaN,None,NaN,8.3,140.0,None,5.7,None
2,10384,438990028,2AZY7HU,-128.90,3.0,53.4,3.0,1.1,NaN,2026-03-23T02:00:00,...,1018.0,None,NaN,None,NaN,8.3,180.0,None,2.6,None
3,10384,438997757,2AZY7HU,-128.90,3.0,53.4,3.0,1.1,NaN,2026-03-23T03:00:00,...,1017.5,None,NaN,None,NaN,8.3,130.0,None,6.2,None
4,10384,439004876,2AZY7HU,-128.90,3.0,53.4,3.0,1.1,NaN,2026-03-23T04:00:00,...,1018.3,None,NaN,None,NaN,8.3,150.0,None,6.2,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17781,10384,439115574,ZZB5AKK,0.48,NaN,51.5,NaN,NaN,56.0,2026-03-23T19:00:00,...,1020.1,None,NaN,None,NaN,57.0,NaN,None,NaN,None
17782,10384,439122596,ZZB5AKK,0.48,NaN,51.5,NaN,NaN,56.0,2026-03-23T20:00:00,...,1020.3,None,NaN,None,NaN,57.0,NaN,None,NaN,None
17783,10384,439129529,ZZB5AKK,0.48,NaN,51.5,NaN,NaN,56.0,2026-03-23T21:00:00,...,1020.5,None,NaN,None,NaN,57.0,NaN,None,NaN,None
17784,10384,439137159,ZZB5AKK,0.48,NaN,51.5,NaN,NaN,56.0,2026-03-23T22:00:00,...,1020.4,None,NaN,None,NaN,57.0,NaN,None,NaN,None


## Do Quality Control Checks

We use `marine_qc` to do some quality control checks on each individual report.

* We start with some positional check.
* Then we do a datime check existing of a date and a time check
* Afterwards we do some checks on the observed sea surface temperature. This is a hard limit check, a missing value check and a freeze check
* These checks will be performed at once using a quality control dictionary containing all relevant inforamtion.
* The the end we do some check on two observed variables. This is a supersaturation and a wind consistency check.

### do_positional_check

We start with a postition check. This check tests whether latitude and longitude values are within valid ranges.

-90 <= latitude <= 90

-180 <= longitude <= 180

In [12]:
pos_qc = do_position_check(
    lat=df["geogr_breite"],
    lon=df["geogr_laenge"],
)

Now we plot the lines corresponding to failed positions (the flag for failed is `1`).

In [13]:
df[pos_qc == 1][["geogr_laenge", "geogr_breite"]]

,geogr_laenge,geogr_breite
4379,117.80,236.68
5971,449.72,22.10
14261,374.36,-4.40


Obviously, these are invalid latitude and/or longitude values.

### do_date_check, do_time_check

Now, we focus on the datetime. Firsly, we have to convert the strings representing datetimes to real datetime objects.

In [16]:
df["messzeit_dt"] = pd.to_datetime(df["messzeit"])
df["messzeit_dt"]

0       2026-03-23 00:00:00
1       2026-03-23 01:00:00
2       2026-03-23 02:00:00
3       2026-03-23 03:00:00
4       2026-03-23 04:00:00
                ...        
17781   2026-03-23 19:00:00
17782   2026-03-23 20:00:00
17783   2026-03-23 21:00:00
17784   2026-03-23 22:00:00
17785   2026-03-23 23:00:00
Name: messzeit_dt, Length: 17786, dtype: datetime64[ns]

In [18]:
date_qc = do_date_check(
    date=df["messzeit_dt"],
    year_init=2000,
    year_end=2030,
)
date_qc

0        0
1        0
2        0
3        0
4        0
        ..
17781    0
17782    0
17783    0
17784    0
17785    0
Length: 17786, dtype: int64

In [19]:
df[date_qc == 1]["messzeit_dt"]

Series([], Name: messzeit_dt, dtype: datetime64[ns])

The date check shows that all dates are valid. This is not suprising since an equivalent tests was done before providing the data. 
Reports with invalid dates are deselected.

Now, let's have a look at the times.

In [21]:
time_qc = do_time_check(
    date=df["messzeit_dt"],
)
time_qc

0        0
1        0
2        0
3        0
4        0
        ..
17781    0
17782    0
17783    0
17784    0
17785    0
Length: 17786, dtype: int64

In [22]:
df[time_qc == 1]["messzeit_dt"]

Series([], Name: messzeit_dt, dtype: datetime64[ns])

As expected, the result is equal to the date check.

### do_missing_value_check

A pre-processing step is to convert the the seas surface temperature from `°C` to `K` .

In [23]:
df["wassertemperatur_kelvin"] = df["wassertemperatur"] + 273.15
df["wassertemperatur_kelvin"]

0        283.75
1           NaN
2           NaN
3           NaN
4           NaN
          ...  
17781       NaN
17782       NaN
17783       NaN
17784       NaN
17785       NaN
Name: wassertemperatur_kelvin, Length: 17786, dtype: float64

Now, let's flag all missing observed values as failed.

In [25]:
miss_qc = do_missing_value_check(
    df["wassertemperatur_kelvin"]
)
miss_qc

0        0
1        1
2        1
3        1
4        1
        ..
17781    1
17782    1
17783    1
17784    1
17785    1
Length: 17786, dtype: int64

In [26]:
df[miss_qc == 1]["wassertemperatur_kelvin"]

1       NaN
2       NaN
3       NaN
4       NaN
6       NaN
         ..
17781   NaN
17782   NaN
17783   NaN
17784   NaN
17785   NaN
Name: wassertemperatur_kelvin, Length: 13316, dtype: float64

In [27]:
miss_qc.value_counts()

1    13316
0     4470
Name: count, dtype: int64

The main part is missing data.

### do_hard_limit_check

Let's flag sea surface temperatures less than `-4.0 °C` and more than `45.0 °C` as failed.

In [29]:
limit_qc = do_hard_limit_check(
    df["wassertemperatur_kelvin"],
    limits=[-4.0 + 273.15, 45.0 + 273.15],
)
limit_qc

0        0
1        2
2        2
3        2
4        2
        ..
17781    2
17782    2
17783    2
17784    2
17785    2
Length: 17786, dtype: int64

In [31]:
df[limit_qc == 1]["wassertemperatur_kelvin"]

11334    268.65
Name: wassertemperatur_kelvin, dtype: float64

We get expected results again.

### do_sst_freeze_check

Let's flag observed sea surface temperatures below the freezing point of sea water (`-1.8 °C`) as failed.

In [33]:
freeze_qc = do_sst_freeze_check(
    df["wassertemperatur_kelvin"],
    freezing_point=-1.8 + 273.15,
)
freeze_qc

0        0
1        2
2        2
3        2
4        2
        ..
17781    2
17782    2
17783    2
17784    2
17785    2
Length: 17786, dtype: int64

In [34]:
df[freeze_qc == 1]["wassertemperatur_kelvin"]

4308     270.95
11334    268.65
Name: wassertemperatur_kelvin, dtype: float64

### do_multiple_individual_check

We can apply `do_hard_limit_check` and `do_sst_freeze_check` with one call using `do_multiple_individual_check`.

Therfor, we need a Quality Control dictionary containig the relevant information.

In [35]:
qc_dict = {
    "HARD": {
        "func": "do_hard_limit_check",
        "names": {"value": "wassertemperatur_kelvin"},
        "arguments": {"limits": [-4.0 + 273.15, 45.0 + 273.15]},
    },
    "FREEZE": {
        "func": "do_sst_freeze_check",
        "names": {"sst": "wassertemperatur_kelvin"},
        "arguments": {"freezing_point": -1.8 + 273.15},
    }
}

Let's do the checks. As soon as a test fails, the subsequent ones are no longer run (`return_method='failed'`).

In [36]:
df_sst_qc = do_multiple_individual_check(
    df,
    qc_dict,
    return_method="failed",
)
df_sst_qc

,HARD,FREEZE
0,0,0
1,2,2
2,2,2
3,2,2
4,2,2
...,...,...
17781,2,2
17782,2,2
17783,2,2
17784,2,2


We got a DataFrame containing QC flags for each test:

* `0`: passed
* `1`: failed
* `2`: not checked (this is for missing values)

We write a little helper function to get one over-all QC flag.

In [37]:
import numpy as np

def get_single_qc_flag(df):
    """Get single QC flag from DataFrame containing multiple QC flags."""
    mask_0 = (df == 0).any(axis=1)
    mask_1 = (df == 1).any(axis=1)
    mask_2 = (df == 2).any(axis=1)
    mask_3 = (df == 3).any(axis=1)

    conditions = [mask_1, mask_0, mask_3, mask_2]
    choices = [1, 0, 3, 2]
    result = np.select(conditions, choices, default=2)
    return pd.Series(result, index=df.index, name="QC_FLAG")

In [38]:
sst_qc = get_single_qc_flag(df_sst_qc)
sst_qc

0        0
1        2
2        2
3        2
4        2
        ..
17781    2
17782    2
17783    2
17784    2
17785    2
Name: QC_FLAG, Length: 17786, dtype: int64

In [39]:
df[sst_qc == 1]["wassertemperatur_kelvin"]

4308     270.95
11334    268.65
Name: wassertemperatur_kelvin, dtype: float64

In [40]:
df[sst_qc == 2]["wassertemperatur_kelvin"]

1       NaN
2       NaN
3       NaN
4       NaN
6       NaN
         ..
17781   NaN
17782   NaN
17783   NaN
17784   NaN
17785   NaN
Name: wassertemperatur_kelvin, Length: 13316, dtype: float64

The results are as expected.

### do_supersaturation_check, do_wind_consistency_check

Finally, we do soem checks that compare two observed variables.

We take air temperature and dew point temperature to tests supersaturation. We convert both of them from `°C` to `K`.

In [42]:
df["lufttemperatur_kelvin"] = df["lufttemperatur"] + 273.15 
df["lufttemperatur_kelvin"]

0        274.15
1        274.75
2        275.05
3        275.35
4        275.55
          ...  
17781    285.05
17782    284.35
17783    284.05
17784    283.45
17785    283.35
Name: lufttemperatur_kelvin, Length: 17786, dtype: float64

In [43]:
df["taupunkttemperatur_kelvin"] = df["taupunkttemperatur"] + 273.15
df["taupunkttemperatur_kelvin"]

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
          ...  
17781    276.75
17782    277.15
17783    278.05
17784    279.15
17785    279.65
Name: taupunkttemperatur_kelvin, Length: 17786, dtype: float64

In [44]:
super_qc = do_supersaturation_check(
    at2 = df["lufttemperatur_kelvin"],
    dpt = df["taupunkttemperatur_kelvin"],
)
super_qc

0        2
1        2
2        2
3        2
4        2
        ..
17781    0
17782    0
17783    0
17784    0
17785    0
Length: 17786, dtype: int64

In [45]:
df[super_qc == 1][["lufttemperatur_kelvin", "taupunkttemperatur_kelvin"]]

,lufttemperatur_kelvin,taupunkttemperatur_kelvin
140,270.75,270.85
7555,279.15,286.45
14690,271.15,271.35
14691,271.15,271.35
16621,297.45,299.05
16622,297.35,298.95
16623,297.45,299.05
16624,297.45,299.05
16625,297.35,298.95
16626,297.45,299.05


Supersaturation is when dew point temperature is higher than air temperature.

The last check is a wind consistency check. 

Zero windspeed should correspond to no particular direction (variable) and wind speeds above a threshold should correspond to a particular direction.

In [46]:
wind_qc = do_wind_consistency_check(
    wind_speed = df["windgeschwindigkeit"],
    wind_direction = df["windrichtung"]
)
wind_qc

0        0
1        0
2        0
3        0
4        0
        ..
17781    2
17782    2
17783    2
17784    2
17785    2
Length: 17786, dtype: int64

In [47]:
df[wind_qc == 1][["windgeschwindigkeit", "windrichtung"]]

,windgeschwindigkeit,windrichtung
1037,0.0,360.0
1039,0.0,360.0
1040,0.0,360.0
1042,0.0,360.0
1044,0.0,360.0
1045,0.0,360.0
1050,0.0,360.0
1051,0.0,360.0
1052,0.0,360.0
1053,0.0,360.0


If wind speed is `0`, wind direction should be `0` too.